In [ ]:
import pickle
from typing import Literal
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root
import tqdm
import tqdm.auto
tqdm.auto.tqdm = tqdm.tqdm
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import collections

from python.config import load_config

cfg = load_config(os.environ.get('CONFIG', 'config.json'))

import python.plots as plots
from python.error_metrics import LoadedErrorMetrics, FilteredErrorMetrics, MetricsIndex as MI # Alias
from python.sampled_hierarchies import SampledHierarchies, SingleStepHierarchies, PartitionStepHierarchies
from python.statistical_test import slope_ttest, page_trend_test, CombineMethod, wilcoxon_test, TrendType, sign_test, per_series_test
from python.aggregation import Aggregation

from itertools import product

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [ ]:
AVAILABLE_LEVELS = Aggregation.complete_summing_matrix(n=cfg.LEVELS.SUBSET, seed=42)

if cfg.is_single_step_hierarchies():
    if cfg.LEVELS.PARTITION == 0:
        sh = SingleStepHierarchies(AVAILABLE_LEVELS)
    else:
        sh = PartitionStepHierarchies(AVAILABLE_LEVELS, cfg.LEVELS.PARTITION, cfg.LEVELS.BASE_PERIOD)
else:
    sh = SampledHierarchies(AVAILABLE_LEVELS, cfg.LEVELS.DIFF, cfg.LEVELS.SEQUENCES, max_h=cfg.LEVELS.MAX_H)

datasets = ['sim', 'web-traffic', 'M4', 'M5']
models = ['AutoETS', 'AutoARIMA', 'Prophet']

ems = {}

for d, m in product(datasets, models):
    ID = f"{d}/{m}"
    em = FilteredErrorMetrics(LoadedErrorMetrics(f"metrics/{cfg.MASTER_FOLDER}/{ID}/sample_hiers_errors.csv"), MI(error_metric='nrmse'))
    ems[ID] = em

In [ ]:
results = {}


for d, m in tqdm.tqdm(product(datasets, models)):
    ID = f"{d}/{m}"
    tqdm.tqdm.write(ID)
    em = ems[ID]

    zero = 1e-6
    results[ID] = {}
    for level in [('Top'), ('Bottom')]:
        results[ID][level] = {}
        for test_fun, test_name in [(wilcoxon_test, 'wilcoxon'), (sign_test, 'sign')]:
            results[ID][level][test_name] = {
                'decr':  per_series_test(test_fun, sh, em, granularity=level, error_metric='nrmse', trend_type=TrendType.DECREASING),
                'incr':  per_series_test(test_fun, sh, em, granularity=level, error_metric='nrmse', trend_type=TrendType.INCREASING),
                'stag':  per_series_test(test_fun, sh, em, granularity=level, error_metric='nrmse', trend_type=TrendType.STAGNANT)
            }


with open(f"metrics/{cfg.MASTER_FOLDER}/per_series_test_results.pkl", 'wb') as f:
    pickle.dump(results, f)


In [ ]:
with open(f"metrics/{cfg.MASTER_FOLDER}/per_series_test_results.pkl", 'rb') as f:
    results: dict = pickle.load(f)

In [ ]:
rows = []
ALPHA = 0.05

for d in datasets:
    for m in models:
        for g in ['Top', 'Bottom']:
            em = ems[f'{d}/{m}']
            reco_minus_base = (em.get_error(MI(hierarchy='Minimal', forecast_type='reconciled_forecasts', granularity=g)) -
                               em.get_error(MI(hierarchy='Minimal', forecast_type='base_forecasts', granularity=g)).values).T
            reco_minus_base = reco_minus_base[reco_minus_base.columns[0]]
            for test in ['sign', 'wilcoxon']:
                row: dict = {
                        'dataset': d,
                        'model': m,
                        'granularity': g,
                        'test': test,
                }
                n_series = len(reco_minus_base)
                def percentage_true(array) -> float:
                    return array.sum() / n_series

                test_res = results[f'{d}/{m}'][g][test]
                row['fraction_decr'] = percentage_true(test_res['decr'].per_unit['p_value'].le(ALPHA))
                row['fraction_incr'] = percentage_true(test_res['incr'].per_unit['p_value'].le(ALPHA))
                row['fraction_stag'] = percentage_true(test_res['stag'].per_unit['p_value'].gt(ALPHA))

                adherences = test_res['decr'].per_unit['p_value'].le(ALPHA) | test_res['stag'].per_unit['p_value'].gt(ALPHA)
                improvements = reco_minus_base <= 0



                row['P(adherence i.e. decr union stag)'] = percentage_true(adherences)
                row['P(improvement i.e. reco_e<=base_e)'] = percentage_true(improvements)

                row['P(improvement ∩ adherence)'] = percentage_true(improvements & adherences)
                row['p(improvement ∩ ~adherence)'] = percentage_true(improvements & ~adherences)

                row['P(adherence | improvement)'] = percentage_true(improvements & adherences) / percentage_true(improvements)

                rows.append(row)

df = pd.DataFrame(rows)
df = df.set_index(keys=['granularity', 'test', 'dataset', 'model']).sort_index()
per_dataset_test_results = df
df.to_csv('per_dataset_test_results.csv', index=True)
df


In [ ]:
test = 'wilcoxon'

for col in [
'P(adherence i.e. decr union stag)',
'P(adherence | improvement)'
]:
    for g in ('Top', 'Bottom'):
        df = per_dataset_test_results.loc[g, test, :, :][col]
        total_weight = 0
        weighted_sum = 0
        for d, s in [('M4', 4000), ('M5', 31000), ('web-traffic', 13000)]:
            weighted_sum += df.loc[(d, slice(None))].mean() * s
            total_weight += s

        print(col, g, test, weighted_sum / total_weight)


In [ ]:
raw_dfs = []

for d in datasets:
    for m in models:
        ID = f"{d}/{m}"
        em = ems[ID]
        test_res = results[ID]
        df = pd.DataFrame(index=em.ts_columns)
        df['orig_bottom'] = em.get_error(MI(hierarchy='Minimal', forecast_type='reconciled_forecasts', granularity='Bottom')).T
        df['orig_top'] =    em.get_error(MI(hierarchy='Minimal', forecast_type='reconciled_forecasts', granularity='Top')).T

        expanded = df.loc[df.index.repeat(len(sh.hierarchies))]
        expanded['aggregations'] = np.tile([str(aggs[1:-1]) for aggs in sh.hierarchy_level_names.values()], len(em.ts_columns))
        expanded['hierarchy'] = np.tile([str(h) for h in sh.hierarchies], len(em.ts_columns))
        for h in sh.hierarchies:
            mask = expanded['hierarchy'] == str(h)
            b = em.get_error(MI(hierarchy=h, forecast_type='reconciled_forecasts', granularity='Bottom')).T.iloc[:, 0]
            t = em.get_error(MI(hierarchy=h, forecast_type='reconciled_forecasts', granularity='Top')).T.iloc[:, 0]
            expanded.loc[mask, 'new_bottom'] = expanded.index[mask].map(b)
            expanded.loc[mask, 'new_top']    = expanded.index[mask].map(t)

        expanded['data'] = ID
        expanded.index.name = 'series_id'
        expanded = expanded.set_index(['data', 'hierarchy'], append=True)

        raw_dfs.append(expanded)

raw_error_values = pd.concat(raw_dfs, axis=0)

In [ ]:
raw_error_values

In [ ]:
raw_error_values.to_csv('raw_error_values.csv', index=True)

In [ ]:
comprehensive_dfs = []

for d in datasets:
    for m in models:
        ID = f"{d}/{m}"
        em = ems[ID]
        test_res = results[ID]
        df = pd.DataFrame(index=em.ts_columns)
        df.index.name = 'series_id'

        df['data'] = ID

        df['orig_bottom'] = em.get_error(MI(hierarchy='Minimal', forecast_type='reconciled_forecasts', granularity='Bottom')).T
        df['orig_top']    = em.get_error(MI(hierarchy='Minimal', forecast_type='reconciled_forecasts', granularity='Top')).T

        df['improvement_bot'] = df['orig_bottom'] < em.get_error(MI(hierarchy='Minimal', forecast_type='base_forecasts', granularity='Bottom')).T.values.flatten()
        df['improvement_top'] = df['orig_top']    < em.get_error(MI(hierarchy='Minimal', forecast_type='base_forecasts', granularity='Top')).T.values.flatten()

        df['le%_bot'] = (em.get_error(MI(hierarchy='Minimal', forecast_type='reconciled_forecasts', granularity='Bottom')).values >= em.get_error(MI(hierarchy=sh.hierarchies, forecast_type='reconciled_forecasts', granularity='Bottom')).values
                         ).sum(axis=0) / len(sh.hierarchies)
        df['gt%_bot'] = 1 - df['le%_bot']

        df['le%_top'] = (em.get_error(MI(hierarchy='Minimal', forecast_type='reconciled_forecasts', granularity='Top')).values >= em.get_error(MI(hierarchy=sh.hierarchies, forecast_type='reconciled_forecasts', granularity='Top')).values
                         ).sum(axis=0) / len(sh.hierarchies)
        df['gt%_top'] = 1 - df['le%_top']

        df['new_bot_mean'] = em.get_error(MI(hierarchy=sh.hierarchies, forecast_type='reconciled_forecasts', granularity='Bottom')).mean(axis=0).values
        df['new_top_mean'] = em.get_error(MI(hierarchy=sh.hierarchies, forecast_type='reconciled_forecasts', granularity='Top')).mean(axis=0).values

        df['new_bot_median'] = em.get_error(MI(hierarchy=sh.hierarchies, forecast_type='reconciled_forecasts', granularity='Bottom')).median(axis=0).values
        df['new_top_median'] = em.get_error(MI(hierarchy=sh.hierarchies, forecast_type='reconciled_forecasts', granularity='Top')).median(axis=0).values

        for test in ('sign', 'wilcoxon'):
            for g in ('Bottom', 'Top'):
                test_res = results[f'{d}/{m}'][g][test]
                for trend in ('decr', 'incr', 'stag'):
                    df[f'{test}_{trend}_{g}_p'] = test_res[trend].per_unit['p_value']

        comprehensive_dfs.append(df)

comprehensive_df = pd.concat(comprehensive_dfs, axis=0)

In [ ]:
comprehensive_df

In [ ]:
comprehensive_df.to_csv('per_series_stats.csv', index=True)

In [ ]:

TEST: Literal['sign', 'wilcoxon'] = 'wilcoxon'
GRANULARITY: Literal['Top', 'Bottom'] = 'Bottom'
ALPHA = 0.05

sig = pd.DataFrame(index=comprehensive_df.index)
sig['data'] = comprehensive_df['data']
sig['improvement'] = comprehensive_df['improvement_bot' if GRANULARITY == 'Bottom' else 'improvement_top'].map({True: 'improved', False: 'not improved'})
sig['increasing']  = comprehensive_df[f'{TEST}_incr_{GRANULARITY}_p'] < ALPHA
sig['decreasing']  = comprehensive_df[f'{TEST}_decr_{GRANULARITY}_p'] < ALPHA
sig['stagnant']    = comprehensive_df[f'{TEST}_stag_{GRANULARITY}_p'] > ALPHA


def category(row) -> str:
    parts = [name for name in ('decreasing', 'stagnant', 'increasing') if row[name]]
    return ' + '.join(parts) if parts else 'none'


sig['category'] = sig.apply(category, axis=1)

counts = sig.groupby(['data', 'improvement', 'category']).size().rename('count').reset_index()
totals = counts.groupby(['data', 'improvement'])['count'].transform('sum')
counts['fraction'] = counts['count'] / totals

color_map = {
    'decreasing':              '#2ca02c',  # green
    'decreasing + stagnant':   '#17becf',  # green+blue blend (teal)
    'stagnant':                '#1f77b4',  # blue
    'stagnant + increasing':   '#9467bd',  # blue+red blend (purple)
    'increasing':              '#d62728',  # red
    'decreasing + increasing': '#8c564b',  # green+red blend (brown, should not occur)
    'none':                    '#c7c7c7',  # grey
}
category_order = [c for c in color_map if c in counts['category'].values]

counts['category'] = pd.Categorical(counts['category'], categories=category_order[::-1], ordered=True)
counts = counts.sort_values(['data', 'improvement', 'category'])


improvement_order = ['improved', 'not improved']
frac = counts.set_index(['data', 'improvement', 'category'])['fraction'].to_dict()
data_order = sorted(counts['data'].unique())

base = collections.defaultdict(float)  # running stack height per (data, improvement)
fig = go.Figure()
for cat in category_order:              # outer loop fixes stack order (bottom first)
    for imp in improvement_order:
        xs, ys, bs = [], [], []
        for d in data_order:
            y = frac.get((d, imp, cat), 0.0)
            if y == 0.0:
                continue
            xs.append(d); ys.append(y); bs.append(base[(d, imp)])
            base[(d, imp)] += y
        if not xs:
            continue
        fig.add_bar(
            x=xs, y=ys, base=bs,
            name=cat, legendgroup=cat,
            showlegend=(imp == improvement_order[0]),
            marker_color=color_map[cat],
            offsetgroup=imp,
            customdata=[imp] * len(xs),
            hovertemplate='%{x} — %{customdata}<br>' + cat + ': %{y:.1%}<extra></extra>',
        )
fig.update_layout(
    barmode='group',
    yaxis_tickformat='.0%',
    xaxis_tickangle=-60,
    title=f'Per-series trend significance ({TEST}, {GRANULARITY}) — left: improved, right: not improved by reconciliation',
    xaxis_title='dataset / model',
    yaxis_title='fraction of series',
    legend_title='detected trend(s)',
    height=700,
)
fig.add_hline(y=0.5, line_dash='dash', line_color='black')
fig.show()